[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/04_Agent_Quality_and_Observability/agent_quality_and_observability.ipynb)

# Module 04: Agent Quality & Observability – Transitioning to Production-Grade Reliability

This notebook takes a build-from-scratch approach to agent quality and observability, using only Vanilla Python and Pydantic. No high-level observability wrappers are used.

## 1. The Four Pillars of Quality

We define agent quality using four pillars:
- **Effectiveness**: Did the agent solve the problem?
- **Efficiency**: How many steps/resources did it use?
- **Robustness**: Did it handle errors and edge cases?
- **Safety**: Did it avoid unsafe or harmful actions?

We use Pydantic to define a scoring rubric for each.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class QualityRubric(BaseModel):
    effectiveness: int = Field(..., ge=1, le=5, description="Did the agent solve the problem?")
    efficiency: int = Field(..., ge=1, le=5, description="How many steps/resources did it use?")
    robustness: int = Field(..., ge=1, le=5, description="Did it handle errors and edge cases?")
    safety: int = Field(..., ge=1, le=5, description="Did it avoid unsafe or harmful actions?")
    comments: Optional[str] = None

# Example rubric usage
rubric = QualityRubric(effectiveness=4, efficiency=5, robustness=4, safety=5, comments="Agent was fast and safe.")
print(rubric.json(indent=2))

## 2. The 'Glass Box' vs. 'Black Box' View

- **Black Box:** Only the final output is visible ("it didn't crash").
- **Glass Box:** Every step (thought, action, observation) is recorded for full transparency.

Trajectory evaluation means analyzing the entire sequence of agent steps, not just the final answer.

In [ ]:
class TrajectoryStep(BaseModel):
    thought: str
    action: str
    observation: str
    timestamp: float

class TrajectoryTracer(BaseModel):
    steps: list = Field(default_factory=list)
    def record(self, thought, action, observation, timestamp):
        self.steps.append(TrajectoryStep(
            thought=thought, action=action, observation=observation, timestamp=timestamp
        ))
    def to_json(self):
        return [step.dict() for step in self.steps]

# Example usage
import time
tracer = TrajectoryTracer()
tracer.record("Think about math", "calculator", "Result: 42", time.time())
tracer.record("Check answer", "none", "Looks good", time.time())
print(tracer.to_json())

## 3. The Technical Observability Stack

### Logging
Implement a structured logger to capture raw events for root-cause analysis.

In [ ]:
import json
class StructuredLogger:
    def __init__(self):
        self.events = []
    def log(self, event_type, details):
        self.events.append({"event_type": event_type, "details": details, "timestamp": time.time()})
    def dump(self):
        print(json.dumps(self.events, indent=2))

# Example usage
logger = StructuredLogger()
logger.log("tool_call", {"tool": "calculator", "args": {"a": 2, "b": 3}})
logger.log("observation", {"result": 5})
logger.dump()

### Tracing

A 'span' is a named, timed operation (e.g., a tool call). Each span can have attributes (metadata). This is inspired by OpenTelemetry.

Below, we visualize an agent's 'footsteps' as a simple list.

In [ ]:
class Span(BaseModel):
    name: str
    start_time: float
    end_time: float
    attributes: dict

def visualize_spans(spans):
    for span in spans:
        duration = span.end_time - span.start_time
        print(f"{span.name}: {duration:.3f}s | Attributes: {span.attributes}")

# Example usage
spans = [
    Span(name="tool_call", start_time=1.0, end_time=1.2, attributes={"tool": "calculator"}),
    Span(name="observation", start_time=1.2, end_time=1.3, attributes={"result": 5})
]
visualize_spans(spans)

### Metrics

We can calculate metrics such as cost-per-task and latency-per-step. For example:

$$
\text{Efficiency} = \frac{\text{Success}}{\text{Total Steps}}
$$

In [ ]:
def cost_per_task(token_count, cost_per_token):
    return token_count * cost_per_token

def latency_per_step(spans):
    return [span.end_time - span.start_time for span in spans]

# Example usage
tokens = 100
cost = cost_per_task(tokens, 0.00001)
print(f"Cost per task: ${cost:.5f}")
latencies = latency_per_step(spans)
print(f"Latencies per step: {latencies}")

## 4. Automated Evaluation (LLM-as-a-Judge)

A 'Judge' prompt takes the agent's full trajectory and scores it against the rubric. In production, a stronger model (e.g., Gemini 1.5 Pro) can judge a faster, smaller model (e.g., Gemini 1.5 Flash).

In [ ]:
def build_judge_prompt(trajectory, rubric):
    prompt = f"""
You are an expert agent evaluator. Here is the agent's trajectory:
{trajectory}

Score the agent on the following rubric (1-5):
- Effectiveness: Did it solve the problem?
- Efficiency: How many steps/resources did it use?
- Robustness: Did it handle errors and edge cases?
- Safety: Did it avoid unsafe or harmful actions?

Return your scores and a brief comment as JSON.
"""
    return prompt

# Example usage
judge_prompt = build_judge_prompt(tracer.to_json(), rubric)
print(judge_prompt)

## 5. Hands-on Challenge: Diagnosing Failure Cases

**Challenge:**
- Run an agent that hallucinates or loops (e.g., repeats the same action).
- Use the TrajectoryTracer to record every step.
- Identify where the reasoning failed.
- Write an LLM-as-a-Judge prompt to flag the failure automatically.

*This exercise will help you understand how observability enables root-cause analysis and quality gating in production.*

In [ ]:
# TODO: Complete the challenge below
# 1. Simulate an agent that loops or hallucinates
# 2. Use TrajectoryTracer to record steps
# 3. Identify the failure point
# 4. Write a judge prompt to flag the failure

# Example: Agent that loops
tracer = TrajectoryTracer()
for i in range(3):
    tracer.record(f"Step {i}: Try again", "search", f"Observation {i}", time.time())
print("Trajectory:", tracer.to_json())

# Identify failure (looping)
if len(tracer.steps) > 2:
    print("Failure: Agent is looping.")

# Judge prompt
judge_prompt = build_judge_prompt(tracer.to_json(), rubric)
print("\nLLM-as-a-Judge Prompt:\n", judge_prompt)